# P49 — QLoRA: ajuste fino eficiente de modelos cuantizados

## 1. Título y paper

**Paper:** *QLoRA: Efficient Finetuning of Quantized LLMs*  
**Autoría:** Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, Luke Zettlemoyer  
**Año y venue:** 2023 · arXiv:2305.14314 · NeurIPS 2023  
**Nivel:** L3 · **Motor:** `quantization`  
**Ficha completa:** [`P49_qlora`](../../papers/foundational/P49_qlora/README.md)

**Hito:** Pone el ajuste fino de un modelo muy grande al alcance de una sola GPU de consumo.

- [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: LoRA reduce los parámetros entrenables, pero el modelo base seguía teniendo que caber en memoria en precisión alta: eso dejaba fuera a casi todo el mundo.
2. Ejecutar una implementación mínima de la propuesta: Cuantizar el modelo base congelado a 4 bits con un formato adaptado a la distribución de los pesos, y entrenar encima adaptadores LoRA en precisión alta.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P48


## 4. Intuición

Guardar cada peso con menos decimales. Suena a pérdida garantizada, y lo es — pero mucho menor de lo que parece, porque los pesos se agrupan en un rango estrecho y no hace falta tanta precisión para distinguirlos.


## 5. Concepto mínimo

```text
16 bits → 65 536 niveles      140 GB para un modelo de 70 000 M
 4 bits →     16 niveles       35 GB para el mismo modelo

QLoRA:  base cuantizada a 4 bits y CONGELADA + adaptadores LoRA en precisión alta
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('quantization', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánta memoria ahorra pasar de 16 a 4 bits?
2. ¿Qué le pasa al error de cuantización?
3. ¿Por qué los adaptadores van en precisión alta?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('quantization', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('quantization', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Pasar de 16 a 4 bits divide la memoria por cuatro, y el error de reconstrucción crece pero se mantiene pequeño frente a la escala de los pesos. Los adaptadores van en precisión alta porque son la parte que **se entrena**: ahí el gradiente sí necesita resolución.


## 10. Comentario pedagógico

El error de reconstrucción de los pesos **no es** el error del modelo. Un modelo puede tolerar mucho ruido en pesos poco influyentes y muy poco en otros. Por eso la cuantización se valida midiendo calidad en tareas, nunca por el error numérico.


## 11. Error o anti-patrón deliberado

Anti-patrón: elegir el número de bits mirando solo el error de reconstrucción.


In [ ]:
print('error de pesos bajo ≠ modelo igual de bueno')
print('  · unos pocos pesos atipicos dominan el resultado y se cuantizan mal')
print('  · la degradacion aparece en tareas concretas, no en la media')
print('  · hay que medir en la tarea, no en la norma del error')

## 12. Corrección

El protocolo mínimo para aceptar una cuantización:


In [ ]:
protocolo = {'medir': 'la tarea real, no la perplejidad sola',
             'comparar': 'contra el modelo sin cuantizar, mismo prompt y semilla',
             'buscar': 'degradacion concentrada en casos raros, no solo la media',
             'reportar': 'bits, formato, que capas se dejaron sin cuantizar'}
show(protocolo)

## 13. Desafío guiado

Calcula cuánta VRAM necesitas para un modelo de 70 000 M a 4, 8 y 16 bits, y con cuál cabe en 24 GB.


In [ ]:
r = run_paper_lab('quantization', seed=3)['result']
show(r)

## 14. Desafío autónomo

Cuantiza un modelo abierto pequeño a 8 y 4 bits y compara su calidad en una tarea concreta, no solo la perplejidad. Busca casos donde la degradación sea desproporcionada.


## 15. Evidencia de aprendizaje

Guarda la tabla de bits, error y memoria, y tu protocolo de aceptación de una cuantización.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P49_qlora/README.md) · evaluación formal: [`assessments/papers/P49_qlora.md`](../../assessments/papers/P49_qlora.md)


## 16. Cierre

El modelo ya cabe y se adapta barato. Queda decidir cuándo un modelo es **aceptable**.


## 17. Conexión con el siguiente hito

- ecosistema local de modelos abiertos

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
